# Step 2: Feature engineering
Code is in `src/features.py`. Only 2021-2025 is explored; 2026 rows are built but not summarised.

In [1]:
import sys, os; ROOT=os.path.expanduser('~/hundred-momentum'); sys.path.insert(0, ROOT+'/src')
import numpy as np, pandas as pd, time
from features import build
B = pd.read_parquet(ROOT+'/data/clean/balls.parquet')
raw = B[[c for c in B if not c.endswith('_pre') and c not in ('ball_no','phase','legal','is_wicket')]]
t=time.time(); D, feats = build(raw); print(len(feats),'features', D.shape, f'{time.time()-t:.1f}s')

80 features (38777, 113) 1.1s


## Leakage test
Pick a ball. Scramble the outcome of that ball, every later ball in the match, and every match on the same or a later date. Rebuild. No feature at or before that ball may change. The same test is run on a deliberately leaky feature to prove it can fail.

In [2]:
rng = np.random.default_rng(0)
OUT = ['runs_bat','runs_extra','runs_total','wides','noballs','byes','legbyes','n_wickets','bowler_wicket']
base = D.reset_index(drop=True); src = raw.sort_values(['date','match_id','innings','seq']).reset_index(drop=True)
dev = np.where((base.season <= 2025) & (base.date > base.date.iloc[0]))[0]; bad = set(); caught = 0; picks = rng.choice(dev, 12, replace=False)
for p in picks:
    r = src.iloc[p]; same = (src.match_id == r.match_id)
    later = (src.date >= r.date) & ~same | same & ((src.innings > r.innings) | ((src.innings == r.innings) & (src.seq >= r.seq)))
    s2 = src.copy(); ix = np.where(later)[0]; s2.loc[ix, OUT] = src.loc[rng.permutation(ix), OUT].values
    D2, _ = build(s2); D2 = D2.reset_index(drop=True)
    cmp = (src.date < r.date) | same & ((src.innings < r.innings) | ((src.innings == r.innings) & (src.seq <= r.seq)))
    c = np.where(cmp)[0]
    for f in feats:
        a, b = base[f].values[c].astype(float), D2[f].values[c].astype(float)
        if not np.allclose(a, b, equal_nan=True, atol=1e-9): bad.add(f)
    leak = lambda d: d.groupby(['match_id','innings']).runs_total.cumsum()
    caught += not np.allclose(leak(base).values[c], leak(D2).values[c])
print('cut points:', len(picks), '| features that changed:', sorted(bad), '| leaky control caught at', caught, 'of', len(picks))
assert not bad

cut points: 12 | features that changed: [] | leaky control caught at 9 of 12


## Suspicion check: no single feature should explain much of one ball

In [3]:
V = D[(D.season<=2025)&(D.full_match==1)]
cc = V[feats].corrwith(V.runs_total).abs().sort_values(ascending=False); print(cc.head(12).round(3).to_string()); print('max |corr| with runs:', round(cc.max(), 3))
print('missing %:'); m=(V[feats].isna().mean()*100).round(1); print(m[m>0].to_string())

bat_runs        0.103
new_batter      0.100
bat_balls       0.100
part_runs       0.087
last5_wkt       0.085
part_balls      0.082
since_wicket    0.082
score_pre       0.080
last10_wkt      0.080
last10_runs     0.077
last5_runs      0.077
last20_runs     0.073
max |corr| with runs: 0.103
missing %:
crr                         1.1
runs_needed                52.0
req_rate                   52.0
rate_gap                   52.5
chase_target               52.0
last5_runs                  1.0
last5_wkt                   1.0
last5_dot                   1.0
last5_bnd                   1.0
last10_runs                 1.0
last10_wkt                  1.0
last10_dot                  1.0
last10_bnd                  1.0
last20_runs                 1.0
last20_wkt                  1.0
last20_dot                  1.0
last20_bnd                  1.0
prev_runs                   1.0
bat_sr                      8.2
bat_sr10                   22.0
bat_dots10                 22.0
bowl_econ                

In [4]:
D.to_parquet(ROOT+'/data/clean/features.parquet', index=False); import json; json.dump(feats, open(ROOT+'/data/clean/feature_list.json','w')); print('saved', D.shape)

saved (38777, 113)
